## One Hot Encoding - sklearn



- OneHotEncoder
- OneHotEncoder + ColumnTransformer
- OneHotEncoder + SimpleImputer in a pipeline

We will see how to perform one hot encoding with Scikit-learn using the Titanic dataset.



In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split

# sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [44]:
# load titanic dataset

usecols = ["pclass", "sibsp", "parch", "sex", "embarked", "cabin", "survived"]

data = pd.read_csv("../../Datasets/titanic.csv", usecols=usecols)
data["cabin"] = data["cabin"].str[0]

data.head()

,pclass,survived,sex,sibsp,parch,cabin,embarked
0,1,1,female,0,0,B,S
1,1,1,male,1,2,C,S
2,1,0,female,1,2,C,S
3,1,0,male,1,2,C,S
4,1,0,female,1,2,C,S


### Encoding important

Just like imputation, all methods of categorical encoding should be performed over the training set, and then propagated to the test set. 

Why? 

Because these methods will "learn" patterns from the train data, and therefore you want to avoid leaking information and overfitting. But more importantly, because we don't know whether in future / live data, we will have all the categories present in the train data, or if there will be more or less categories. Therefore, we want to anticipate this uncertainty by setting the right processes right from the start. We want to create transformers that learn the categories from the train set, and used those learned categories to create the dummy variables in both train and test sets.

In [45]:
# let's separate into training and testing set

X_train, X_test, y_train, y_test = train_test_split(
    data.drop("survived", axis=1),  # predictors
    data["survived"],  # target
    test_size=0.3,  # percentage of obs in test set
    random_state=0,
)  # seed to ensure reproducibility

X_train.shape, X_test.shape

((916, 6), (393, 6))

## One hot encoding with Scikit-learn

### Advantages

- quick
- Creates the same number of features in train and test set
- works within a pipeline

### Limitations

- need to set specific output to return pandas
- need additional transformer to encode variable subset
- changes variable names after transformation

In [46]:
# To start, we fillna (Adding "Missing") manually. Later on
# we add this step to a pipeline

X_train.fillna("Missing", inplace=True)
X_test.fillna("Missing", inplace=True)
X_train

,pclass,sex,sibsp,parch,cabin,embarked
501,2,female,0,1,Missing,S
588,2,female,1,1,Missing,S
402,2,female,1,0,Missing,C
1193,3,male,0,0,Missing,Q
686,3,female,0,0,Missing,Q
...,...,...,...,...,...,...
763,3,female,1,2,Missing,S
835,3,male,0,0,Missing,S
1216,3,female,0,0,Missing,Q
559,2,female,0,0,Missing,S


In [47]:
# set up encoder

encoder = OneHotEncoder(
    categories="auto",
    drop="first",  #"first" to return k-1, (use drop=None to return k dummies)
    sparse_output=False,
    handle_unknown="error",  # helps deal with rare labels
)
#encoder
encoder.set_output(transform="pandas")

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [48]:
# fit the encoder (finds categories)

encoder.fit(X_train)

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [49]:
# we observe the learned categories

encoder.categories_

[array([1, 2, 3]),
 array(['female', 'male'], dtype=object),
 array([0, 1, 2, 3, 4, 5, 8]),
 array([0, 1, 2, 3, 4, 5, 6, 9]),
 array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'Missing', 'T'], dtype=object),
 array(['C', 'Missing', 'Q', 'S'], dtype=object)]

In [50]:
# transform the data sets

X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)

#X_train_enc.head()
#X_train
X_train_enc

#X_train_enc.to_excel("X_train_encoded.xlsx", index=False) generate excel file for reference


,pclass_2,pclass_3,sex_male,sibsp_1,sibsp_2,sibsp_3,sibsp_4,sibsp_5,sibsp_8,parch_1,...,cabin_C,cabin_D,cabin_E,cabin_F,cabin_G,cabin_Missing,cabin_T,embarked_Missing,embarked_Q,embarked_S
501,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
588,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
402,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1193,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
686,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
835,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1216,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
559,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [51]:
# we can retrieve the feature names as follows:

encoder.get_feature_names_out()

array(['pclass_2', 'pclass_3', 'sex_male', 'sibsp_1', 'sibsp_2',
       'sibsp_3', 'sibsp_4', 'sibsp_5', 'sibsp_8', 'parch_1', 'parch_2',
       'parch_3', 'parch_4', 'parch_5', 'parch_6', 'parch_9', 'cabin_B',
       'cabin_C', 'cabin_D', 'cabin_E', 'cabin_F', 'cabin_G',
       'cabin_Missing', 'cabin_T', 'embarked_Missing', 'embarked_Q',
       'embarked_S'], dtype=object)

## Encoding variable subset

In [52]:
# et up encoder

encoder = OneHotEncoder(
    categories="auto",
    drop=None,  # to return k-1, use drop=None to return k dummies
    sparse_output=False,
    handle_unknown="error",  # helps deal with rare labels
)

In [53]:
# select the variables to encode

ct = ColumnTransformer(
    [("encoder", encoder, ["sex", "embarked", "cabin"])], remainder="passthrough"
)

ct.set_output(transform="pandas")

,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [54]:
# train encoder

ct.fit(X_train)

,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [55]:
# encode

X_train_enc = ct.transform(X_train)
X_test_enc = ct.transform(X_test)

X_train_enc.head()

,encoder__sex_female,encoder__sex_male,encoder__embarked_C,encoder__embarked_Missing,encoder__embarked_Q,encoder__embarked_S,encoder__cabin_A,encoder__cabin_B,encoder__cabin_C,encoder__cabin_D,encoder__cabin_E,encoder__cabin_F,encoder__cabin_G,encoder__cabin_Missing,encoder__cabin_T,remainder__pclass,remainder__sibsp,remainder__parch
501,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2,0,1
588,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2,1,1
402,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2,1,0
1193,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3,0,0
686,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3,0,0


## imputation and encoding

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop("survived", axis=1),  # predictors
    data["survived"],  # target
    test_size=0.3,  # percentage of obs in test set
    random_state=0,  # seed to ensure reproducibility
)

X_train.shape, X_test.shape

((916, 6), (393, 6))

In [57]:
# check for missing data

X_train.isnull().sum()

pclass        0
sex           0
sibsp         0
parch         0
cabin       702
embarked      2
dtype: int64

In [58]:
# set up encoder
encoder = OneHotEncoder(
    categories="auto",
    drop=None,  # "first" to return k-1, use drop=None to return k dummies
    sparse_output=False,
    handle_unknown="error",  # helps deal with rare labels
)

In [59]:
# set up encoder and imputation in pipeline
# we only want to impute categorical variables

pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        (("ohe", encoder)),
    ]
)

In [60]:
# select the variables to transform (impute + encode)

ct = ColumnTransformer(
    [("encoder", pipe, ["sex", "embarked", "cabin"])], remainder="passthrough"
)

ct.set_output(transform="pandas")

,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'constant'
,fill_value,'missing'


In [61]:
# fit pipeline

ct.fit(X_train)

,transformers,"[('encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'constant'
,fill_value,'missing'


In [62]:
# transform data

X_train_enc = ct.transform(X_train)
X_test_enc = ct.transform(X_test)

X_train_enc.head()

,encoder__sex_female,encoder__sex_male,encoder__embarked_C,encoder__embarked_Q,encoder__embarked_S,encoder__embarked_missing,encoder__cabin_A,encoder__cabin_B,encoder__cabin_C,encoder__cabin_D,encoder__cabin_E,encoder__cabin_F,encoder__cabin_G,encoder__cabin_T,encoder__cabin_missing,remainder__pclass,remainder__sibsp,remainder__parch
501,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2,0,1
588,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2,1,1
402,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2,1,0
1193,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,0,0
686,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,0,0


In [63]:
# the size of the resulting dataframes is identical

X_train_enc.shape, X_test_enc.shape

((916, 18), (393, 18))